<a href="https://colab.research.google.com/github/shauryasachdev/Vizuara_CV/blob/main/AlexNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from tensorflow.keras import layers, models

alexnet = models.Sequential([
    layers.Input(shape=(227, 227, 3)),  # Input size used in original paper (227x227x3 for RGB images)

    # Conv1: 96 filters, 11x11 kernel, stride 4
    layers.Conv2D(96, kernel_size=11, strides=4, activation='relu'),

    # Max Pool1: 3x3 pool, stride 2
    layers.MaxPooling2D(pool_size=3, strides=2),

    # Conv2: 256 filters, 5x5 kernel, same padding
    layers.Conv2D(256, kernel_size=5, padding='same', activation='relu'),

    # Max Pool2: 3x3 pool, stride 2
    layers.MaxPooling2D(pool_size=3, strides=2),

    # Conv3: 384 filters, 3x3 kernel, same padding
    layers.Conv2D(384, kernel_size=3, padding='same', activation='relu'),

    # Conv4: 384 filters, 3x3 kernel, same padding
    layers.Conv2D(384, kernel_size=3, padding='same', activation='relu'),

    # Conv5: 256 filters, 3x3 kernel, same padding
    layers.Conv2D(256, kernel_size=3, padding='same', activation='relu'),

    # Max Pool3: 3x3 pool, stride 2
    layers.MaxPooling2D(pool_size=3, strides=2),

    # Flatten the output for fully connected layers
    layers.Flatten(),

    # FC1: 4096 units, ReLU
    layers.Dense(4096, activation='relu'),

    # Dropout for regularization
    layers.Dropout(0.5),

    # FC2: 4096 units, ReLU
    layers.Dense(4096, activation='relu'),

    # Dropout for regularization
    layers.Dropout(0.5),

    # FC3: 1000 units (for ImageNet classes), Softmax
    layers.Dense(1000, activation='softmax')
])

# Print the model summary
alexnet.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 55, 55, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 27, 27, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 27, 27, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 13, 13, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 13, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1000)           │     4,097,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,378,344 (237.95 MB)

 Trainable params: 62,378,344 (237.95 MB)

 Non-trainable params: 0 (0.00 B)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import wandb
import os

In [4]:
# ===========================
# STEP 0: Initializing wandb
# ===========================

wandb.init(project="alexnet-flowers-v4", config ={
    "epochs":10,
    "batch_size":16,
    "learning_rate":0.001,
    "architecture":"alexnet",
    "pretrained": True,
    "Input_size":224
})

config = wandb.config

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shauryasachdev to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
import pathlib
import numpy as np

# =============================================
# Step 1: Data Preparation
# =============================================

transform = transforms.Compose([
    transforms.Resize((config.Input_size, config.Input_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])


# Your exact path as Path object
data_dir = pathlib.Path('/content/drive/MyDrive/Colab Notebooks/alexnet_flowers_dataset/')
train_dir = pathlib.Path('/content/drive/MyDrive/Colab Notebooks/alexnet_flowers_dataset/train')
val_dir = pathlib.Path('/content/drive/MyDrive/Colab Notebooks/alexnet_flowers_dataset/val')

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)


In [6]:
# =============================================
# Step 2: Load Pre-Trained Model (CORRECT ORDER — THIS IS THE FIX)
# =============================================

# Modern way (no deprecation warning + future-proof)
weights = models.AlexNet_Weights.IMAGENET1K_V1 if config.pretrained else None
alexnet = models.alexnet(weights=weights)

# 1. Replace the final layer FIRST
alexnet.classifier[6] = nn.Linear(4096, 5)

# 2. Freeze everything except the new head
for param in alexnet.parameters():
    param.requires_grad = False
for param in alexnet.classifier[6].parameters():
    param.requires_grad = True

# 3. Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alexnet = alexnet.to(device)

# 4. NOW create optimizer (key fix!)
optimizer = optim.Adam(alexnet.classifier[6].parameters(), lr=0.001)

# Loss function
criterion = nn.CrossEntropyLoss()

# W&B monitoring
wandb.watch(alexnet, log="all", log_freq=10)

print("✅ Model ready with CORRECT optimizer order!")
print(f"   Device: {device}")
print(f"   Trainable parameters: {sum(p.numel() for p in alexnet.parameters() if p.requires_grad)} (only the final layer)")

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 165MB/s]


✅ Model ready with CORRECT optimizer order!
   Device: cpu
   Trainable parameters: 20485 (only the final layer)


In [7]:
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10):
  for epoch in range(epochs):
    model.train()
    train_correct = 0
    train_total = 0
    running_loss = 0

    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-" * 30)

    for i, (images, labels) in enumerate(train_loader):
      images, labels = images.to(device), labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      _, preds = torch.max(outputs, 1)
      batch_correct = (preds == labels).sum().item()
      train_correct += batch_correct
      train_total += labels.size(0)

      # Print every 10 batches
      if (i+1) % 10 == 0:
        batc_acc = batch_correct / labels.size(0)
        print(f"[Batch {i+1}/{len(train_loader)}] Loss: {loss.item():.4f} Batch Accuracy: {batc_acc:.4f}")

    train_acc = train_correct / train_total
    wandb.log({"epoch": epoch + 1, "train_loss": running_loss, "train_accuracy": train_acc})
    print(f"Epoch {epoch+1} Summary - Loss: {running_loss:.4f}, Train Accuracy: {train_acc:.4f}")


    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total
    wandb.log({"epoch": epoch + 1, "val_accuracy": val_acc})
    print(f"Validation Accuracy: {val_acc:.4f}")

In [8]:
# ===================
# Train the model
# ===================
train_model(alexnet, criterion, optimizer, train_loader, val_loader, epochs=config.epochs)


Epoch 1/10
------------------------------
[Batch 10/251] Loss: 0.9367 Batch Accuracy: 0.7500
[Batch 20/251] Loss: 0.4470 Batch Accuracy: 0.8750
[Batch 30/251] Loss: 0.4473 Batch Accuracy: 0.8125
[Batch 40/251] Loss: 0.9213 Batch Accuracy: 0.6875
[Batch 50/251] Loss: 0.6402 Batch Accuracy: 0.8125
[Batch 60/251] Loss: 0.2927 Batch Accuracy: 0.9375
[Batch 70/251] Loss: 0.7238 Batch Accuracy: 0.8750
[Batch 80/251] Loss: 0.2822 Batch Accuracy: 0.8750
[Batch 90/251] Loss: 1.1629 Batch Accuracy: 0.7500
[Batch 100/251] Loss: 0.4816 Batch Accuracy: 0.8125
[Batch 110/251] Loss: 0.5268 Batch Accuracy: 0.8125
[Batch 120/251] Loss: 0.4057 Batch Accuracy: 0.8750
[Batch 130/251] Loss: 0.3003 Batch Accuracy: 0.8750
[Batch 140/251] Loss: 0.6331 Batch Accuracy: 0.8125
[Batch 150/251] Loss: 0.9163 Batch Accuracy: 0.6875
[Batch 160/251] Loss: 0.2533 Batch Accuracy: 0.9375
[Batch 170/251] Loss: 0.5627 Batch Accuracy: 0.8125
[Batch 180/251] Loss: 0.3499 Batch Accuracy: 0.8125
[Batch 190/251] Loss: 0.6433 B